In [26]:
import sys
import numpy as np
import pandas as pd
import sklearn
print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__ )

3.12.7 2.0.2 2.3.3 1.6.1


In [27]:
from sklearn.model_selection import train_test_split

SEED = 2653

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


# part 1
## 1. Parent entropy

there are 12 pationts
6 p. have disease = 1
6 p. have disease = 0

p1 = 6/12 = 0.5
p2 = 6/12 = 0.5

H (parent) = -0.5 * log_2(0.5) - 0.5 * log_2(0.5)
           = -0.5 * (-1) - 0.5 * (-1) = 0.5 + 0.5 = 1 bit


## 2. infotmation gain
### 2.1 age 55 plus

Split:
age_55_plus  = 1: 6 p. , 5 disease, 1 healthy

p1 = 5/6, p0 = 1/6
H = -(5/6) * log_2(5/6) - (1/6) * log_2(1/6) = 0.650 bit

age_55_plus = 0: 6p. , 1 disease, 5 healthy

p1 = 1/6, p0 = 5/6
H = -(1/6) * log_2(1/6) - (5/6) * log_2(5/6) = 0.650 bit

weighted entropy = (6/12) * 0.650 + (6/12) * 0.650 = 0.650

IG = 1 - 0.650 = 0.350

### 2.2 exercise_angina

split:
exercise_angina = 1: 4p. , 4 diseased, 0 healthy

p1 = 4/4, p0 = 0/4
H = -1 * log_2(1) - 0 * log_2(0) = 0 bit

exercise_angina = 0: 8p. , 2 diseased, 6 healthy

p1 = 2/8, p0 = 6/8
H = -0.25 * log_2(0.25) - 0.75 * log_2(0.75) = 0.811 bit

w.entropy = 4/12 * 0 - 8/12 * 0.811 = 0.541

### 2.3 male

split:
male = 1: 6p. , 3 diseased, 3 healthy

p1 = 3/6, p0 = 3/6
H = -(3/6) * log_2(3/6) - (3/6) * log_2(3/6) = 1 bit

male = 0: 6p. , 3 diseased, 3 healthy (the same)
H = 1 bit

IG = 1 - 1 = 0 bit  (male doen't help)

### 2.4 hight_bp

split:
high_bp = 1: 6p. , 4 deseased, 2 healthy

p1 = 4/6, p0 = 2/6
H = -(4/6) * log_2(4/6) - (2/6) * log_2(2/6) = 0.918 bit

high_bp = 0: 6p. , 2 diseased, 4 healthy

p1 = 2/6, p0 = 4/6
H = -(2/6) * log_2(2/6) - (4/6) * log_2(4/6) = 0.918 bit

W.entropy = 0.5 * 0.918 + 0.5 * 0.918 = 0.918

IG = 1 - 0.918 = 0.082 bit

## 3 root
                    IG

age_55_plus =     0.350
exercise_angina = 0.459
male =            0.0
high_bp =         0.082

root of the tree "exercise_angina". It has a highest IG

## 4. inside the right branch

split:
age_55_plus = 1: 2p. , 1 d, 1h

p1 = 1/2, p0 = 1/2
H = -0.5 * log_2(0.5) - 0.5 * log(0.5) = 1

age_55_plus = 0: 6p. , 1d, 5h

p1 = 1/6, p0 = 5/6
H = -(1/6) * log_2(1/6) - (5/6) * log_2(5/6) = 0.650

w.entr = (2/8) * 1 + (6/8) * 0.650 = 0.738
IG = 0.811 - 0.738 = 0.073


split:
male = 1: 3p. , 0d, 3h

p1 = 0/3, p0 = 3/3
H = 0 * log_2(0) - 1 * log_2(1) = 0

male = 0: 5p. , 2d, 3h

p1 = 2/5, p0 = 3/5
H = -(2/5) * log_2(2/5) - (3/5) * log_2(3/5) = 0.971

w.entr = (3/8) * 0 + (5/8) * 0.971 = 0.607
IG = 0.811 - 0.607 = 0.204


split:
high_bp = 1: 3p. , 1d, 2h

p1 = 1/3, p0 = 2/3
H = -(1/3) * log_2(1/3) - (2/3) * log_2(2/3) = 0.918

high_bp = 0: 5p. , 1d, 4h

p1 = 1/5, p0 = 4/5
H = -(1/5) * log_2(1/5) - (4/5) * log_2(4/5) = 0.722

w.entr = (3/8) * 0.918 + (5/8) * 0.722 = 0.795
IG = 0.811 - 0.795 = 0.016


IG:
male = 0.204
age_55_plus = 0.073
high_bp = 0.016

chosen feature inside the branch iy's "male"


# Part 2

In [2]:
def entropy(y):
    """ Entropy in bits of an array of integer class labels."""
    if len(y) == 0:
        return 0.0

    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return -np.sum(probs * np.log2(probs, where=probs > 0)) 

In [11]:
toy = pd.DataFrame({
    "age_55_plus":     [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "exercise_angina": [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "male":            [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "high_bp":         [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    "disease":         [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

print("H(parent) =", entropy(toy["disease"].to_numpy()))


left  = toy["exercise_angina"] == 1   
right = toy["exercise_angina"] == 0   
print("H(exercise_angina=1) =", entropy(toy.loc[left,  "disease"].to_numpy()))
print("H(exercise_angina=0) =", entropy(toy.loc[right, "disease"].to_numpy()))

H(parent) = 1.0
H(exercise_angina=1) = -0.0
H(exercise_angina=0) = 0.8112781244591328


In [9]:
def information_gain(y, left):
    """ Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    H_parent = entropy(y)
    H_left = entropy(y[left])
    H_right = entropy(y[~left])
    w_left = np.sum(left) / len(y)
    w_right = np.sum(~left) / len(y)
    w_entropy = w_left * H_left + w_right * H_right
    return H_parent - w_entropy

In [12]:
y = toy["disease"].to_numpy()

print("H(parent) =", entropy(y))
print()

left = (toy["age_55_plus"] == 1).to_numpy()
print(f"age_55_plus:     IG = {information_gain(y, left):.3f}")

left = (toy["exercise_angina"] == 1).to_numpy()
print(f"exercise_angina: IG = {information_gain(y, left):.3f}")

left = (toy["male"] == 1).to_numpy()
print(f"male:            IG = {information_gain(y, left):.3f}")

left = (toy["high_bp"] == 1).to_numpy()
print(f"high_bp:         IG = {information_gain(y, left):.3f}")

H(parent) = 1.0

age_55_plus:     IG = 0.350
exercise_angina: IG = 0.459
male:            IG = 0.000
high_bp:         IG = 0.082


In [15]:
def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best = None
    best_gain = -1.0

    for feature_idx in range(X.shape[1]):
        values = X[:, feature_idx]
        thresholds = (np.unique(values)[:-1] + np.unique(values)[1:]) / 2

        for threshold in thresholds:
            left = values <= threshold

            if np.sum(left) < min_samples_leaf:
                continue
            if np.sum(~left) < min_samples_leaf:
                continue

            gain = information_gain(y, left)

            if gain > best_gain:
                best_gain = gain
                best = (feature_idx, float(threshold), float(gain))

    return best

In [16]:
X_toy = toy.drop(columns="disease").to_numpy()
y_toy = toy["disease"].to_numpy()

result = best_split(X_toy, y_toy)
print(result)

(1, 0.5, 0.4591479170272448)


In [34]:
def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts."""

    if len(np.unique(y)) == 1:
        return {"leaf": int(y[0])}

    if max_depth is not None and depth >= max_depth:
        majority = int(np.argmax(np.bincount(y)))
        return {"leaf": majority}

    split = best_split(X, y, min_samples_leaf=min_samples_leaf)

    if split is None:
        majority = int(np.argmax(np.bincount(y)))
        return {"leaf": majority}


    feature_idx, threshold, _ = split
    left_mask = X[:, feature_idx] <= threshold

    left_child  = build_tree(X[left_mask],  y[left_mask],
                             max_depth=max_depth,
                             min_samples_leaf=min_samples_leaf,
                             depth=depth + 1)
    right_child = build_tree(X[~left_mask], y[~left_mask],
                             max_depth=max_depth,
                             min_samples_leaf=min_samples_leaf,
                             depth=depth + 1)

    return {
        "feature": feature_idx,
        "threshold": float(threshold),
        "left": left_child,
        "right": right_child,
    }

In [38]:
def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    predictions = []
    for row in X:
        node = tree
        while "leaf" not in node:
            feature_idx = node["feature"]
            threshold = node["threshold"]
            if row[feature_idx] <= threshold:
                node = node["left"]
            else:
                node = node["right"]
        predictions.append(node["leaf"])
    return np.array(predictions)

In [32]:
from sklearn.tree import DecisionTreeClassifier

ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED)
ref.fit(X_train, y_train)

print("sklearn feature:", feature_names[ref.tree_.feature[0]], "threshold:", ref.tree_.threshold[0])

our_split = best_split(X_train, y_train)
print("our best_split:", (feature_names[our_split[0]], our_split[1], our_split[2]))

sklearn feature: ca threshold: 0.5
our best_split: ('ca', 0.5, 0.18324542662794518)


In [39]:
depths = [1, 2, 3, 4, 5, None]
results = []

for d in depths:
   
    our_tree = build_tree(X_train, y_train, max_depth=d)
    our_preds = predict_tree(our_tree, X_test)

   
    sk_tree = DecisionTreeClassifier(
        criterion="entropy", max_depth=d, random_state=SEED
    ).fit(X_train, y_train)
    sk_preds = sk_tree.predict(X_test)

    agreement = (our_preds == sk_preds).mean()
    results.append((d, agreement))

for d, a in results:
    label = "None" if d is None else str(d)
    print(f"max_depth={label:>4}  agreement={a:.4f}")

max_depth=   1  agreement=1.0000
max_depth=   2  agreement=1.0000
max_depth=   3  agreement=1.0000
max_depth=   4  agreement=1.0000
max_depth=   5  agreement=0.9444
max_depth=None  agreement=1.0000
